# Spatial quantification and plotting

Plot allocentric head direction, bearings towards fixed or moving ROIs, and spatial occupancy.

Set the recording inputs, load the streams, then run the angle and occupancy sections. Each recording uses its own clock, calibration and background image. Pose processing runs before trial selection.


In [ ]:
#=== 1| Import session, measurement and plotting tools ========
from pathlib import Path
import sys

# Find src/ from the checkout root or any notebook subdirectory.
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / "movement_figures").is_dir():
        sys.path.insert(0, str(parent))
        break
    if (parent / "src" / "movement_figures").is_dir():
        sys.path.insert(0, str(parent / "src"))
        break
else:
    raise RuntimeError("Start Jupyter inside the data-conduit checkout.")

import matplotlib.pyplot as plt
import movement
import movement.kinematics as kin
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from movement.plots import plot_occupancy
from movement.utils.vector import compute_signed_angle_2d

from data_conduit.datastructures import slice_stream, slice_stream_for_trial
from data_conduit.refactor_qc.training_filter import filter_trials
from movement_figures.data_template.loading import (
    build_datastructure,
    prepare_pose,
)
from data_conduit.integrations.DLC.pose import pose_to_movement
from movement_figures.video import read_session_video_frame
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries

print(f"movement {movement.__version__}: {movement.__file__}")


## Recording, processing and selection settings

`INCLUDE_SESSIONS` and `LEVEL_SELECTORS` choose recordings. `TRIAL_RANGE=(1, 8)` selects original trials 1–8 within each; `WINDOW` optionally restricts time.

`TRAINING_SPEC` enables the existing training rules. `EXCLUDE_LED_ON_SESSIONS=True` drops any recording with an ON-labelled trial in its original trial table. These labels record ON-event presence, not continuous LED state.

Use the printed recording IDs as calibration/ROI dictionary keys. Fixed ROIs are `(x, y)` pixel positions. Moving ROIs are `(time, space)` DataArrays with `space=["x", "y"]`, matching the full pose clock. Missing ROI entries omit those panels.


In [ ]:
# ===============================================================================
# 2| Choose Recordings, Tracked Landmarks, Processing and Trial Selection
# ===============================================================================

ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")
INCLUDE_SESSIONS = ("2026-06-21T093349Z",)                                    # Add actual recording-folder names, or use None for all selected folders.
LEVEL_NAMES = ("mouseID", "day")
LEVEL_SELECTORS = {}                                                          # Existing l0_selector/l1_selector filters restrict mice and days before reading.
SOURCES = ("trials", "dlc")                                                   # Trial columns annotate angles; DLC also reads its video-timestamp dependency.

INDIVIDUAL = "individual_0"
TRACKING_KEYPOINT = "body"                                                    # Occupancy counts this landmark's positions.
LEFT_EAR, RIGHT_EAR = "lear", "rear"
CAMERA_VIEW = "top_down"                                                      # Movement's ear convention for the camera above the animal.
REFERENCE_VECTOR = (1.0, 0.0)                                                 # Allocentric zero is image-right, unchanged by pixel/cm conversion.

CONFIDENCE_THRESHOLD = 0.9                                                    # Lower-confidence positions become NaN; None disables this operation.
MAX_GAP_FRAMES = None                                                         # None leaves missing samples unfilled; an integer limits the interpolated sample gap.
SMOOTHING_WINDOW = None                                                       # None disables the median; otherwise use an odd positive sample count.

SPATIAL_UNIT = "px"                                                           # Choose "px" or "cm" for positions, image edges and occupancy axes together.
PIXELS_PER_CM_BY_SESSION = {}                                                 # Fill with measured {recording_id: pixels_per_cm} before choosing "cm".
FIXED_ROI_PX_BY_SESSION = {}                                                  # Fill with measured {recording_id: (x, y)} for fixed-ROI bearing panels.
MOVING_ROI_PX_BY_SESSION = {}                                                 # Fill with actual {recording_id: DataArray} on that recording's full pose clock.

TRAINING_SPEC = None                                                          # Optional existing refactor_qc training table; None applies no spreadsheet exclusions.
EXCLUDE_LED_ON_SESSIONS = False                                               # True removes a whole recording containing any original LED-ON trial.
TRIAL_RANGE = None                                                            # Inclusive original trial_index bounds, such as (1, 8); None keeps every remaining trial.
WINDOW = None                                                                 # Optional absolute (start, end) seconds, applied within each recording before plotting.
PLOT_MODE = "scatter"                                                         # "scatter" or "line"; lines break at NaNs, clock gaps and the +/-pi wrap.
TRACE_MAX_GAP_SECONDS = None                                                  # None detects recording gaps above three times the full-clock median interval.


## Load and inspect recordings

Inspect the printed identities and trial table before calculating measurements. Subsequent selections keep the matching recording ID on pose and trial data.


In [ ]:
# ===============================================================================
# 3| Load the Selected Recordings and Apply the Existing Training Rules
# ===============================================================================

# === 1| Configure Directory Selection and Read the Requested Streams ============

data = build_datastructure(
    ROOT,
    level_names=LEVEL_NAMES,
    streams=SOURCES,
    include=INCLUDE_SESSIONS,                                                 # Folder-name selection occurs before the readers open recording files.
    **LEVEL_SELECTORS,
)
streams = data.load()                                                         # Returns the existing stream mapping, not a new figure-result class.
all_trials = streams["trials"]
selected_trials = all_trials.copy()                                           # Keep the original rows available for whole-session LED checks.

# === 2| Apply Optional Training Rules Without Losing Recording Identity ==========

if TRAINING_SPEC is not None:
    session_names = {recording_id: ref.path.name for recording_id, ref in data.sessions.items()}
    selected_trials["session_name"] = selected_trials["session"].map(session_names)  # Add leaf names for the spreadsheet; retain full IDs in session.
    training_rules = TRAINING_SPEC.rename(columns={"session": "session_name"})
    selected_trials = filter_trials(
        selected_trials, training_rules,
        session_column="session_name", report=False,                          # Existing mouse+leaf-name rules; the original recording identity stays intact.
    )

# === 3| Inspect the Loaded Recording IDs and the Trial Rows Available to Select ==

for session_id, session_ref in data.sessions.items():
    print(session_id, "->", session_ref.path)                                 # Use this exact ID as the calibration/ROI dictionary key.
display(selected_trials)


## Allocentric direction and ROI bearings

Allocentric direction is measured from `REFERENCE_VECTOR` to the head-forward vector. The default zero points right; positive angles are clockwise in image coordinates with y down.

Egocentric bearing is the angle from head-forward to **ROI position − ear midpoint**. Zero means straight ahead. A moving ROI's heading is irrelevant. Missing/coincident ears or an ROI at the head centre produce undefined angles.

Choose scatter or line display. Lines break at missing samples, acquisition gaps and the ±π wrap.


In [ ]:
# ===============================================================================
# 4| Draw Already-Selected Head Angles on Their Recording Clock
# ===============================================================================


def plot_head_angles(
    angles: xr.Dataset,                                                       # One recording's selected angles; excluded samples should remain NaN on its clock.
    trials: pd.DataFrame | None = None,                                       # Optional matching trial rows; the function never chooses them.
    *,
    mode: str = "scatter",                                                    # Draw measured points or measured lines, with no additional angle calculation.
    max_gap_seconds: float | None = None,                                     # Caller supplies a gap threshold from the full recording clock.
    title: str = "Head direction",
    color: str = "#5b4b9a",
    marker_size: float = 2.5,
    figsize: tuple[float, float] | None = None,
) -> tuple[Figure, np.ndarray]:
    """
    Draw one panel per supplied angle variable, with optional trial annotations.

    Parameters
    ----------
    angles : xarray.Dataset
        Angle variables in radians, each with only a time dimension containing
        increasing recording-clock seconds. A variable's ``label`` attribute
        supplies its readable axis name. NaNs mark missing or excluded samples.
    trials : pandas.DataFrame | None
        Already-selected trial rows from the same recording and clock. None
        omits the existing Q_C trial annotations. Raw events are not required.
    mode : str
        ``"scatter"`` draws measured points; ``"line"`` joins adjacent valid
        measurements and breaks at NaNs, long clock gaps and jumps above pi.
    max_gap_seconds : float | None
        Positive gap threshold in seconds. None disables clock-gap detection;
        missing samples and circular wrapping still break lines.
    title, color : str
        Figure heading and Matplotlib measurement colour.
    marker_size : float
        Positive scatter-marker size in points. Does not alter measured angles.
    figsize : tuple[float, float] | None
        Figure width/height in inches. None allocates 2.7 inches per angle panel.

    Returns
    -------
    tuple[matplotlib.figure.Figure, numpy.ndarray]
        Figure and one-dimensional axes array. No trial selection, interpolation,
        circular averaging or ``show`` call occurs in this function.
    """

    # === 1| Check the Supplied Angle Clock and Drawing Controls ==================

    if mode not in ("scatter", "line"):
        raise ValueError("mode must be 'scatter' or 'line'.")
    if not angles.data_vars or angles.sizes.get("time", 0) == 0:
        raise ValueError("angles must contain angle variables and time samples.")
    times = np.asarray(angles.time, dtype=float)
    if not np.isfinite(times).all() or np.any(np.diff(times) <= 0):
        raise ValueError("Angle timestamps must be finite and strictly increasing.")
    if max_gap_seconds is not None and (not np.isfinite(max_gap_seconds) or max_gap_seconds <= 0):
        raise ValueError("max_gap_seconds must be finite and positive, or None.")
    if not np.isfinite(marker_size) or marker_size <= 0:
        raise ValueError("marker_size must be finite and positive.")
    for name, angle in angles.data_vars.items():
        if angle.dims != ("time",):
            raise ValueError(f"{name} must have only the time dimension.")

    # === 2| Give Each Angle Its Own Panel with a Shared Recording-Time Axis ======

    if figsize is None:
        figsize = (13, 2.7 * len(angles.data_vars))                           # Increase panel height with the number of supplied ROI measurements.
    fig, grid = plt.subplots(
        len(angles.data_vars), 1, figsize=figsize,
        squeeze=False, sharex=True, layout="constrained",                     # A one-panel plot still returns an indexable axes array.
    )
    axes = grid[:, 0]
    legend_handles = {}

    # === 3| Draw Measurements Without Connecting Through the Circular Wrap ======

    for ax, (name, angle) in zip(axes, angles.data_vars.items()):
        values = np.asarray(angle, dtype=float)
        if mode == "scatter":
            ax.plot(times, values, ".", ms=marker_size, color=color)
        else:
            breaks = np.abs(np.diff(values)) > np.pi                          # A plotted jump from +pi to -pi must not cross through zero.
            if max_gap_seconds is not None:
                breaks |= np.diff(times) > max_gap_seconds                    # Long acquisition gaps are also discontinuous drawing intervals.
            split_positions = np.flatnonzero(breaks) + 1
            break_times = (times[split_positions - 1] + times[split_positions]) / 2
            line_times = np.insert(times, split_positions, break_times)
            line_values = np.insert(values, split_positions, np.nan)          # Keep both endpoint measurements; insert a drawing-only break.
            ax.plot(line_times, line_values, color=color, linewidth=1.0)

        # === 3.1| Label the Angle Convention and Add the Supplied Trial Rows =====

        ax.set_ylabel(angle.attrs.get("label", name) + " (rad)")
        ax.set_ylim(-np.pi, np.pi)
        ax.set_yticks([-np.pi, 0, np.pi], ["−π", "0", "π"])
        ax.grid(axis="y", alpha=0.2)
        if trials is not None and not trials.empty:
            annotations = annotate_qc_timeseries(
                ax, trials, window=(float(times[0]), float(times[-1])),
                legend=False,                                                 # Collect one shared legend after all panels are drawn.
            )
            for handle in annotations.legend_handles:
                legend_handles.setdefault(handle.get_label(), handle)

    # === 4| Return the Labelled Figure and Its Axes to the Caller ================

    axes[-1].set_xlabel("Recording time (s)")
    fig.suptitle(title)
    if legend_handles:
        fig.legend(handles=list(legend_handles.values()), loc="outside lower center", ncol=4)
    return fig, axes


In [ ]:
# ===============================================================================
# 5| Select, Measure and Display Each Recording Separately
# ===============================================================================

spatial_inputs = {}                                                           # Plain selected arrays and trial tables are reused by the later occupancy cell.
head_angles_by_session = {}                                                   # Keep the calculated measurements independently of the figures.
pose_sessions = []                                                            # Optional DLC may be absent from some or all selected recordings.
if "dlc:position" in streams and "dlc:confidence" in streams:
    pose_sessions = pd.unique(streams["dlc:position"].session.values).tolist()

for session_id, session_ref in data.sessions.items():

    # === 1| Apply Whole-Recording LED Exclusion Before the Trial Range ===========

    original_trials = slice_stream(all_trials, selectors={"session": session_id})
    if EXCLUDE_LED_ON_SESSIONS and original_trials["LED"].eq("ON").any():
        print(f"{session_id}: skipped because the original trial table contains LED ON.")
        continue

    trials = slice_stream(selected_trials, selectors={"session": session_id})
    if TRIAL_RANGE is not None:
        trials = slice_stream(trials, selectors={"trial_index": slice(*TRIAL_RANGE)})
    if trials.empty:
        print(f"{session_id}: no trials remain after the selected rules.")
        continue

    if session_id not in pose_sessions:
        print(f"{session_id}: skipped because no aligned pose was loaded.")
        continue

    # === 2| Convert and Process Only This Recording's Aligned DLC Arrays =========

    raw_pose = pose_to_movement(
        slice_stream(streams["dlc:position"], selectors={"session": session_id}),
        slice_stream(streams["dlc:confidence"], selectors={"session": session_id}),
        individual=INDIVIDUAL,                                                # Label the one animal in this DLC export; conversion preserves the acquired clock.
    )
    pose = prepare_pose(
        raw_pose,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        max_gap_frames=MAX_GAP_FRAMES,
        smoothing_window=SMOOTHING_WINDOW,                                    # Filtering uses this full recording; it never joins separate trial slices.
    )
    position_px = pose.position.sel(individual=INDIVIDUAL, drop=True)

    # === 3| Resolve This Recording's Spatial Scale and Selected Time Samples =====

    if SPATIAL_UNIT not in ("px", "cm"):
        raise ValueError("SPATIAL_UNIT must be 'px' or 'cm'.")
    spatial_scale = 1.0
    if SPATIAL_UNIT == "cm":
        pixels_per_cm = PIXELS_PER_CM_BY_SESSION.get(session_id)
        if pixels_per_cm is None or not np.isfinite(pixels_per_cm) or pixels_per_cm <= 0:
            raise ValueError(f"{session_id}: supply a positive measured pixels-per-cm calibration.")
        spatial_scale = 1.0 / float(pixels_per_cm)                            # Apply the same calibration to position, ROI coordinates and image edges.

    keep = xr.zeros_like(position_px.time, dtype=bool)
    for _, trial in trials.iterrows():
        trial_position = slice_stream_for_trial(
            position_px, trial, time_coord="time",                            # The row supplies both recording identity and its numeric time bounds.
        )
        keep |= position_px.time.isin(trial_position.time)                    # The trial row supplies its bounds and endpoint inclusion.
    if WINDOW is not None:
        window_position = slice_stream(position_px, selectors={"time": slice(*WINDOW)})
        keep &= position_px.time.isin(window_position.time)                   # Intersect the trial selection with the explicit clock window.
    if not bool(keep.any()):
        print(f"{session_id}: no pose samples lie inside the selected intervals.")
        continue
    position = (position_px * spatial_scale).where(keep)                      # Retained NaNs stop lines crossing an excluded trial or interval.
    position.attrs["units"] = SPATIAL_UNIT

    # === 4| Define the Head Origin and Reject Missing or Coincident Ears =========

    for landmark in (LEFT_EAR, RIGHT_EAR, TRACKING_KEYPOINT):
        if landmark not in position.keypoint:
            raise ValueError(f"{session_id}: missing tracked keypoint {landmark!r}.")
    reference = np.asarray(REFERENCE_VECTOR, dtype=float)
    if reference.shape != (2,) or not np.isfinite(reference).all() or np.linalg.norm(reference) == 0:
        raise ValueError("REFERENCE_VECTOR must be a finite, nonzero x/y direction.")
    left_ear = position.sel(keypoint=LEFT_EAR, drop=True)
    right_ear = position.sel(keypoint=RIGHT_EAR, drop=True)
    head_centre = (left_ear + right_ear) / 2                                  # Both ROI direction vectors start midway between the ears.
    ear_distance = np.sqrt(((left_ear - right_ear) ** 2).sum("space", skipna=False))
    valid_head = np.isfinite(ear_distance) & (ear_distance > 0)

    # === 5| Call Movement Directly for Allocentric Heading and Head Direction ====

    allocentric = kin.compute_forward_vector_angle(
        position,
        left_keypoint=LEFT_EAR,
        right_keypoint=RIGHT_EAR,
        reference_vector=REFERENCE_VECTOR,                                    # This external direction defines zero, independent of the animal's body.
        camera_view=CAMERA_VIEW,                                              # Top-down ear labels determine which perpendicular direction points forwards.
    ).where(valid_head)
    head_forward = kin.compute_head_direction_vector(
        position,
        left_keypoint=LEFT_EAR,
        right_keypoint=RIGHT_EAR,
        camera_view=CAMERA_VIEW,
    ).where(valid_head)
    head_angles = xr.Dataset({"allocentric_heading": allocentric})
    head_angles.allocentric_heading.attrs["label"] = "Allocentric head direction"

    # === 6| Measure Bearing from the Head Towards the Actual Fixed ROI ===========

    fixed_roi_px = FIXED_ROI_PX_BY_SESSION.get(session_id)
    if fixed_roi_px is not None:
        roi_values = np.asarray(fixed_roi_px, dtype=float)
        if roi_values.shape != (2,) or not np.isfinite(roi_values).all():
            raise ValueError(f"{session_id}: fixed ROI must contain two finite measured pixel coordinates.")
        fixed_roi = xr.DataArray(
            roi_values * spatial_scale, dims="space", coords={"space": ["x", "y"]},
        )
        to_fixed_roi = fixed_roi - head_centre                                # Subtract the current head position, not the animal's body direction.
        fixed_distance = np.sqrt((to_fixed_roi ** 2).sum("space", skipna=False))
        fixed_bearing = compute_signed_angle_2d(head_forward, to_fixed_roi)
        head_angles["fixed_roi_bearing"] = fixed_bearing.where(valid_head & (fixed_distance > 0))
        head_angles.fixed_roi_bearing.attrs["label"] = "Bearing towards fixed ROI"
    else:
        print(f"{session_id}: fixed ROI panel omitted; no measured location supplied.")

    # === 7| Align Actual Moving-ROI Positions and Measure Their Bearing ===========

    moving_roi_px = MOVING_ROI_PX_BY_SESSION.get(session_id)
    if moving_roi_px is not None:
        if not isinstance(moving_roi_px, xr.DataArray) or moving_roi_px.dims != ("time", "space"):
            raise ValueError(f"{session_id}: moving ROI needs dimensions (time, space).")
        if list(moving_roi_px.space.values) != ["x", "y"]:
            raise ValueError(f"{session_id}: moving ROI needs space=['x', 'y'].")
        moving_roi = moving_roi_px * spatial_scale
        moving_roi, aligned_head = xr.align(moving_roi, head_centre, join="exact")  # Reject mismatched clocks; never silently interpolate another animal.
        to_moving_roi = moving_roi - aligned_head                             # Its changing position is relevant; its own heading is not an input.
        moving_distance = np.sqrt((to_moving_roi ** 2).sum("space", skipna=False))
        moving_bearing = compute_signed_angle_2d(head_forward, to_moving_roi)
        valid_roi = np.isfinite(to_moving_roi).all("space") & (moving_distance > 0)
        head_angles["moving_roi_bearing"] = moving_bearing.where(valid_head & valid_roi)
        head_angles.moving_roi_bearing.attrs["label"] = "Bearing towards moving ROI"
    else:
        print(f"{session_id}: moving ROI panel omitted; no aligned position data supplied.")
    for name in head_angles:
        head_angles[name].attrs["units"] = "rad"                              # Uniform position scaling changes distances but not angles.

    # === 8| Plot the Selected Extent and Retain Plain Inputs for Occupancy =======

    selected_times = position.time.where(keep, drop=True)
    selected_extent = slice(float(selected_times.min()), float(selected_times.max()))
    visible_angles = head_angles.sel(time=selected_extent)                    # Trim only leading/trailing time; internal excluded samples remain NaN.
    clock_gaps = np.diff(np.asarray(position.time, dtype=float))
    max_gap_seconds = TRACE_MAX_GAP_SECONDS
    if max_gap_seconds is None and clock_gaps.size:
        max_gap_seconds = 3 * float(np.median(clock_gaps))                    # Estimate ordinary frame spacing before any clock samples are removed.
    angle_figure, angle_axes = plot_head_angles(
        visible_angles, trials,
        mode=PLOT_MODE,
        max_gap_seconds=max_gap_seconds,
        title=f"{session_id} · selected-trial head direction",
    )
    plt.show()
    head_angles_by_session[session_id] = head_angles
    spatial_inputs[session_id] = {
        "position": position,
        "trials": trials,
        "spatial_scale": spatial_scale,
    }                                                                         # No raw clocks or camera coordinates are pooled across these dictionary entries.


## Occupancy of selected trials, outbound and inbound paths

`plot_occupancy` counts valid position samples in rectangular bins—not seconds or probabilities. Empty bins are transparent; bins are not clipped to the arena circle.

Outbound excludes the target sample; inbound includes it. Outer trial inclusion is preserved. Trials lacking a valid target time appear only in the selected-trials panel.

All panels within a recording share bin edges and colour limits. With 30 bins per axis, widths are `(xmax−xmin)/30` and `(ymax−ymin)/30`; the cells need not be square. Their dimensions are printed. `OCCUPANCY_VMAX` changes colour scaling only.

Pose and background image must use the same camera geometry and spatial scale. The `UndistortedVideoData` name alone does not verify lens correction.


In [ ]:
# ===============================================================================
# 6| Control Occupancy Bins, Shared Colour Limits and Matching Video Backgrounds
# ===============================================================================

OCCUPANCY_BINS = 30                                                           # Thirty bins along x and y form 30 x 30 rectangular cells, not a circular mask.
OCCUPANCY_VMAX = 100.0                                                        # Display ceiling in samples/bin; None uses this recording's largest observed count.
OCCUPANCY_ALPHA = 0.70                                                        # Opacity of occupied cells; zero-count cells remain completely transparent.
OCCUPANCY_VIDEO_SUBDIR = "UndistortedVideoData"                               # Select the video coordinate space actually used for DLC tracking.
ARENA_RANGE_PX_BY_SESSION = {}                                                # Optional measured {recording_id: ((xmin, xmax), (ymin, ymax))}; absent means full frame.


In [ ]:
# ===============================================================================
# 7| Overlay Already-Selected Occupancy Arrays on One Matching Video Frame
# ===============================================================================

def plot_spatial_occupancy(
    points: dict[str, xr.DataArray],                                          # Panel names mapped to already-selected positions from one recording.
    background_frame: np.ndarray,                                             # Matching RGB/RGBA image in the same raw/corrected camera space.
    *,
    frame_extent: tuple[float, float, float, float],                          # Image edges after the same pixel/cm conversion as positions.
    spatial_unit: str = "px",
    bins: int = 30,
    arena_range: tuple[tuple[float, float], tuple[float, float]] | None = None,
    vmax: float | None = None,
    alpha: float = 0.70,
    title: str = "Spatial occupancy",
    figsize: tuple[float, float] | None = None,
) -> tuple[Figure, np.ndarray, dict[str, dict[str, np.ndarray]]]:
    """Overlay supplied point arrays on a matching image using rectangular bins.

    Parameters
    ----------
    points : dict[str, xarray.DataArray]
        Panel labels mapped to preselected arrays with dimensions (time, space),
        space=['x', 'y'], and coordinates in spatial_unit. Empty panels are allowed.
    background_frame : numpy.ndarray
        Matching RGB/RGBA video frame, already corrected upstream if required.
    frame_extent : sequence[float]
        Image (left, right, bottom, top) edges in the same units as points;
        image coordinates have bottom > top because y increases downwards.
    spatial_unit : str
        'px' or 'cm'; coordinates and image edges must already be converted.
    bins : int
        Positive number of rectangular bins on each spatial axis.
    arena_range : sequence | None
        Optional ((xmin, xmax), (ymin, ymax)) bounds; None uses the frame edges.
    vmax : float | None
        Colour ceiling in samples/bin. None shares the largest observed count.
    alpha : float
        Overlay opacity between zero and one. Zero-count cells stay transparent.
    title : str
        Overall figure title.
    figsize : tuple[float, float] | None
        Figure width/height in inches. None allocates 5.2 inches per panel.

    Returns
    -------
    tuple[matplotlib.figure.Figure, numpy.ndarray, dict]
        Figure, axes array, and histogram dictionaries for nonempty panels.
        Histograms contain h[x_bin, y_bin], xedges and yedges; counts are unchanged.
        Empty panels still display the matching frame. The function does not
        select trials, rescale positions, correct the video, normalise counts or
        call show. Zero-bin masking and the colour ceiling affect drawing only.
    """
    #=== 1| Check point arrays, image edges and the histogram controls ============

    if not points:
        raise ValueError("Supply at least one named occupancy panel.")
    if spatial_unit not in ("px", "cm"):
        raise ValueError("spatial_unit must be 'px' or 'cm'.")
    if isinstance(bins, (bool, np.bool_)) or not isinstance(bins, (int, np.integer)) or bins < 1:
        raise ValueError("bins must be a positive integer.")
    if vmax is not None and (not np.isfinite(vmax) or vmax <= 0):
        raise ValueError("vmax must be finite and positive, or None.")
    if not np.isfinite(alpha) or not 0 <= alpha <= 1:
        raise ValueError("alpha must be between zero and one.")
    frame = np.asarray(background_frame)
    if frame.ndim != 3 or frame.shape[2] not in (3, 4) or min(frame.shape[:2]) == 0:
        raise ValueError("background_frame must be a nonempty RGB or RGBA image.")
    extent = np.asarray(frame_extent, dtype=float)
    if extent.shape != (4,) or not np.isfinite(extent).all() or extent[0] >= extent[1] or extent[3] >= extent[2]:
        raise ValueError("frame_extent needs increasing x edges and downward image y edges.")
    for label, point in points.items():
        if point.dims != ("time", "space") or list(point.space.values) != ["x", "y"]:
            raise ValueError(f"{label}: expected (time, space) with space=['x', 'y'].")

    # === 2| Give Every Panel the Same Rectangular Bin Edges and Image Extent =====

    # Frame edges are left/right/bottom/top; histogram bounds need low/high x/y.
    # Using these limits keeps empty space in the rectangular camera view rather
    # than shrinking bins to whichever positions happened to be visited.

    limits = np.asarray(arena_range) if arena_range is not None else np.array([
        [extent[0], extent[1]], [extent[3], extent[2]],
    ])
    if limits.shape != (2, 2) or not np.isfinite(limits).all() or np.any(limits[:, 0] >= limits[:, 1]):
        raise ValueError("arena_range must contain increasing finite x and y bounds.")
    if figsize is None:
        figsize = (5.2 * len(points), 5.8)                                    # Keep the same width for each supplied occupancy panel.
    figure, grid = plt.subplots(
        1, len(points), squeeze=False, sharex=True, sharey=True,
        figsize=figsize, layout="constrained",
    )
    axes = grid[0]
    cmap = plt.get_cmap("magma").with_extremes(bad=(0, 0, 0, 0))              # Masked zero-count cells reveal the video completely.
    histograms, meshes = {}, []
    observed_max = 0.0

    #=== 3| Let movement compute each histogram and hide only its empty cells =====

    for ax, (label, point) in zip(axes, points.items()):
        ax.imshow(frame, origin="upper", extent=extent, zorder=0)
        finite = point.where(np.isfinite(point).all("space"), drop=True)      # A histogram sample requires both finite x and y coordinates.
        if finite.sizes["time"]:
            _, _, histogram = plot_occupancy(
                finite, ax=ax, bins=bins, range=limits,
                cmap=cmap, alpha=alpha, zorder=1,                             # Shared binning; opacity changes display, never counts.
            )
            histograms[label] = histogram
            observed_max = max(observed_max, float(histogram["h"].max()))
            mesh = ax.collections[-1]
            if mesh.colorbar is not None:
                mesh.colorbar.remove()                                        # Replace movement's panel keys with one common colourbar.
            mesh.set_array(np.ma.masked_equal(histogram["h"].T, 0))           # Histogram axes are x/y; the drawing uses transposed y/x values.
            meshes.append(mesh)                                               # Histogram axes are x/y; the drawn mesh uses y/x order.
        else:
            ax.text(0.5, 0.5, "No valid samples", transform=ax.transAxes, ha="center")
        ax.set_xlim(*limits[0])
        ax.set_ylim(limits[1, 1], limits[1, 0])                               # Display image y increasing downwards in either unit.
        ax.set_aspect("equal")
        ax.set_title(label)
        ax.set_xlabel(f"x ({spatial_unit})")
    axes[0].set_ylabel(f"y ({spatial_unit})")

    #=== 4| Share the colour ceiling and return untouched histogram counts ========

    ceiling = float(vmax) if vmax is not None else max(1.0, observed_max)
    norm = Normalize(vmin=0, vmax=ceiling)
    for mesh in meshes:
        mesh.set_norm(norm)
    # An independent mappable keeps the colour key opaque while the video overlay
    # remains translucent. A pointed end signals values above a chosen ceiling.
    colourbar = figure.colorbar(
        ScalarMappable(norm=norm, cmap=cmap), ax=list(axes), orientation="horizontal",
        fraction=0.06, pad=0.09, extend="max" if observed_max > ceiling else "neither",
    )
    colourbar.set_label("Valid position samples / bin")
    figure.suptitle(title)
    return figure, axes, histograms


In [ ]:
# ===============================================================================
# 8| Select Outbound/Inbound Samples and Plot Each Recording's Matching Background
# ===============================================================================

occupancy_by_session = {}                                                     # Store raw movement histograms separately from their colour-limited display.

for session_id, selected in spatial_inputs.items():

    # === 1| Recover This Recording's Selected Positions and Original Trial Bounds =

    point = selected["position"].sel(keypoint=TRACKING_KEYPOINT, drop=True)
    trials = selected["trials"]
    spatial_scale = selected["spatial_scale"]
    outbound = xr.zeros_like(point.time, dtype=bool)
    inbound = xr.zeros_like(point.time, dtype=bool)
    valid_trial_count = 0

    # === 2| Split Each Usable Trial at Its Target-Trigger Time ===================

    for _, trial in trials.iterrows():
        start, target, end = trial["start_time"], trial["tz_triggered_time"], trial["end_time"]
        if not np.isfinite([start, target, end]).all() or not start < target < end:
            continue                                                          # A missing or unordered target cannot divide the trial into outbound and inbound portions.

        outbound_trial = trial.copy()
        outbound_trial["end_time"] = target
        outbound_trial["end_inclusive"] = False                               # Target sample goes to inbound; retain the trial's outer start rule.
        inbound_trial = trial.copy()
        inbound_trial["start_time"] = target
        inbound_trial["start_inclusive"] = True                               # Include target; retain the trial's outer end rule.
        outbound_point = slice_stream_for_trial(
            point, outbound_trial, time_coord="time",                         # Match the row to the position array's retained session coordinate.
        )
        inbound_point = slice_stream_for_trial(
            point, inbound_trial, time_coord="time",
        )
        outbound |= point.time.isin(outbound_point.time)
        inbound |= point.time.isin(inbound_point.time)
        valid_trial_count += 1

    occupancy_points = {
        "Selected trials": point,
        "Outbound": point.where(outbound),
        "Inbound": point.where(inbound),
    }                                                                         # The earlier trial/time selection remains NaN even if a portion's numeric bounds cover it.

    # === 3| Read This Recording's Video and Apply Its Measured Spatial Scale =====

    background_frame, video_path = read_session_video_frame(
        data.sessions[session_id].path,
        video_subdir=OCCUPANCY_VIDEO_SUBDIR,                                  # This folder must match the coordinate system used by this recording's DLC.
    )
    height, width = background_frame.shape[:2]
    frame_extent = tuple(np.array([-0.5, width - 0.5, height - 0.5, -0.5]) * spatial_scale)  # Half-pixel edges keep integer tracking coordinates at pixel centres.
    arena_range_px = ARENA_RANGE_PX_BY_SESSION.get(session_id)
    arena_range = None
    if arena_range_px is not None:
        arena_range = tuple(tuple(bounds) for bounds in np.asarray(arena_range_px) * spatial_scale)

    # === 4| Plot the Supplied Portions and Report Their Actual Rectangular Bins ==

    occupancy_figure, occupancy_axes, occupancy = plot_spatial_occupancy(
        occupancy_points, background_frame,
        frame_extent=frame_extent,
        spatial_unit=SPATIAL_UNIT,
        bins=OCCUPANCY_BINS,
        arena_range=arena_range,
        vmax=OCCUPANCY_VMAX,
        alpha=OCCUPANCY_ALPHA,
        title=f"{session_id} · {TRACKING_KEYPOINT} occupancy",
    )
    plt.show()
    occupancy_by_session[session_id] = occupancy
    print(f"{session_id}: {valid_trial_count}/{len(trials)} retained trial rows have usable target times before the optional WINDOW intersection.")
    print("Background:", video_path)
    for label, histogram in occupancy.items():
        bin_width = np.diff(histogram["xedges"])[0]
        bin_height = np.diff(histogram["yedges"])[0]
        print(f"{label}: bin width {bin_width:.3g} {SPATIAL_UNIT}; bin height {bin_height:.3g} {SPATIAL_UNIT}")


Movement reference: [forward-vector angle](https://movement.neuroinformatics.dev/latest/api/movement.kinematics.compute_forward_vector_angle.html), [signed angle](https://movement.neuroinformatics.dev/latest/api/movement.utils.vector.compute_signed_angle_2d.html), [occupancy](https://movement.neuroinformatics.dev/latest/api/movement.plots.plot_occupancy.html).
